# Canonical Model 03: PRT and Parallel Splitting

Build and run the 50 x 50, four-layer canonical DISV model, trace particles with MF6 PRT, and split the same simulation across eight real MPI processes.

> **Validation goal:** every stage must terminate normally, preserve both physical lakes, and produce reconstructable results.


In [1]:
from pathlib import Path
import pandas as pd
import myflopy as mf
from myflopy.modflow.mf6.canonical_example import representative_cells
from canonical_notebook_style import notebook_header

notebook_header('03', 'PRT and Parallel', 'Trace groundwater movement and run the same model across MPI partitions.')

root = Path('../artifacts/canonical_prt_parallel')
config = mf.CanonicalModelConfig.validation()
model = mf.build_canonical_model(root / 'gwf', config=config)
success, gwf_report = model.run_simulation()
assert success, '\n'.join(gwf_report[-30:])

pd.Series({
    'rows': config.nrow,
    'columns': config.ncol,
    'layers': config.nlay,
    'total_3d_cells': config.nrow * config.ncol * config.nlay,
    'gwf_termination': gwf_report[-1],
}, name='canonical GWF')


VoronoiGrid initializing.


Voronoi grid initialized.


Imported 1 features from ..\artifacts\canonical_prt_parallel\gwf\inputs\north_lake.gpkg
Imported 1 features from ..\artifacts\canonical_prt_parallel\gwf\inputs\south_lake.gpkg
Generating connections (rectangular mode) for lake 0


getting connectivity properties (iac, ja, cl12, hwva, nja)


Generating connections (rectangular mode) for lake 1


...

    Solving:  Stress period:     5    Time step:     2


    Solving:  Stress period:     6    Time step:     1


    Solving:  Stress period:     6    Time step:     2


 
 Run end date and time (yyyy/mm/dd hh:mm:ss): 2026/06/09 18:56:21
 Elapsed run time: 29.156 Seconds
 
 Normal termination of simulation.

Success is:  True


rows                                               50
columns                                            50
layers                                              4
total_3d_cells                                  10000
gwf_termination     Normal termination of simulation.
Name: canonical GWF, dtype: object

## MF6 PRT

PRT consumes the completed GWF head, budget, and binary-grid files. Tracking stops at the end of available flow output by default, preventing an unbounded final-time-step run.


In [2]:
releases = mf.PRTReleasePoints.from_cells(model, representative_cells(config)['releases'])
prt = model.particle_tracking.prt(
    workspace=root / 'prt',
    release_points=releases,
    porosity=0.25,
    extend_tracking=False,
)
prt_results = prt.run(silent=False)
assert prt_results.success
assert not prt_results.pathlines.empty

pd.Series({
    'release_points': len(releases.packagedata),
    'pathline_records': len(prt_results.pathlines),
    'tracked_particles': prt_results.pathlines['irpt'].nunique(),
    'track_csv_bytes': prt_results.track_csv_path.stat().st_size,
}, name='PRT results')


writing simulation...
  writing simulation name file...
  writing simulation tdis package...
  writing solution package ems...
  writing model viz_prt_mast_prt...
    writing model name file...
    writing package disv...


    writing package mip...
    writing package prp...
    writing package oc...
    writing package fmi...
FloPy is using the following executable to run the model: ..\..\..\..\..\..\..\mf-env\.venv\Scripts\mf6.exe
                               MODFLOW 6 EXTENDED
                U.S. GEOLOGICAL SURVEY MODULAR HYDROLOGIC MODEL
                        VERSION 6.8.0.dev0+cb8a12e.dirty
                               ***DEVELOP MODE***

   MODFLOW 6 compiled Jun 07 2026 15:32:35 with Intel(R) Fortran Intel(R) 64
  Compiler Classic for applications running on Intel(R) 64, Version 2021.12.0
                             Build 20240222_000000

This software is preliminary or provisional and is subject to 
revision. It is being provided to meet the need for timely best 
science. The software has not received final approval by the U.S. 
Geological Survey (USGS). No warranty, expressed or implied, is made 
by the USGS or the U.S. Government as to the functionality of the 
software and related mat

    Solving:  Stress period:     1    Time step:     1
    Solving:  Stress period:     1    Time step:     2
    Solving:  Stress period:     2    Time step:     1
    Solving:  Stress period:     2    Time step:     2
    Solving:  Stress period:     3    Time step:     1
    Solving:  Stress period:     3    Time step:     2
    Solving:  Stress period:     4    Time step:     1
    Solving:  Stress period:     4    Time step:     2
    Solving:  Stress period:     5    Time step:     1
    Solving:  Stress period:     5    Time step:     2
    Solving:  Stress period:     6    Time step:     1
    Solving:  Stress period:     6    Time step:     2
 
 Run end date and time (yyyy/mm/dd hh:mm:ss): 2026/06/09 18:56:22
 Elapsed run time:  0.574 Seconds
 
 Normal termination of simulation.


release_points           3
pathline_records       203
tracked_particles        3
track_csv_bytes      23328
Name: PRT results, dtype: int64

In [3]:
import pandas as pd
import figs as f

model.gwf.npf.k[0].get_data().reshape((50,50))

fig = f.Fig()

fig.add_heatmap(z=model.gwf.npf.k[0].get_data().reshape((50,50)))

fig.show()

In [4]:
model.hds.map().plot()

In [5]:
model.particle_tracking.open_prt(workspace=r"C:\Users\lukem\Python\Projects\myflopy\examples\mf6\artifacts\canonical_prt_parallel\prt").scene().export_html(path=r"C:\Users\lukem\Python\Projects\myflopy\examples\mf6\artifacts\canonical_prt_parallel\prt\prt.html")

C:\Users\lukem\Python\mf-env\.venv\Lib\site-packages\trame_vtk\modules\vtk\serializers\data.py:43: DeprecationWarning: Call to deprecated method GetData. (Use ExportLegacyFormat, or GetOffsetsArray/GetConnectivityArray instead.) -- Deprecated since version 9.6.0.
  if dataset.GetVerts() and dataset.GetVerts().GetData().GetNumberOfTuples() > 0:
C:\Users\lukem\Python\mf-env\.venv\Lib\site-packages\trame_vtk\modules\vtk\serializers\data.py:49: DeprecationWarning: Call to deprecated method GetData. (Use ExportLegacyFormat, or GetOffsetsArray/GetConnectivityArray instead.) -- Deprecated since version 9.6.0.
  if dataset.GetLines() and dataset.GetLines().GetData().GetNumberOfTuples() > 0:
C:\Users\lukem\Python\mf-env\.venv\Lib\site-packages\trame_vtk\modules\vtk\serializers\data.py:55: DeprecationWarning: Call to deprecated method GetData. (Use ExportLegacyFormat, or GetOffsetsArray/GetConnectivityArray instead.) -- Deprecated since version 9.6.0.
  if dataset.GetPolys() and dataset.GetPolys

C:\Users\lukem\Python\mf-env\.venv\Lib\site-packages\trame_vtk\modules\vtk\serializers\data.py:43: DeprecationWarning: Call to deprecated method GetData. (Use ExportLegacyFormat, or GetOffsetsArray/GetConnectivityArray instead.) -- Deprecated since version 9.6.0.
  if dataset.GetVerts() and dataset.GetVerts().GetData().GetNumberOfTuples() > 0:
C:\Users\lukem\Python\mf-env\.venv\Lib\site-packages\trame_vtk\modules\vtk\serializers\data.py:49: DeprecationWarning: Call to deprecated method GetData. (Use ExportLegacyFormat, or GetOffsetsArray/GetConnectivityArray instead.) -- Deprecated since version 9.6.0.
  if dataset.GetLines() and dataset.GetLines().GetData().GetNumberOfTuples() > 0:
C:\Users\lukem\Python\mf-env\.venv\Lib\site-packages\trame_vtk\modules\vtk\serializers\data.py:55: DeprecationWarning: Call to deprecated method GetData. (Use ExportLegacyFormat, or GetOffsetsArray/GetConnectivityArray instead.) -- Deprecated since version 9.6.0.
  if dataset.GetPolys() and dataset.GetPolys

C:\Users\lukem\Python\mf-env\.venv\Lib\site-packages\trame_vtk\modules\vtk\serializers\data.py:43: DeprecationWarning: Call to deprecated method GetData. (Use ExportLegacyFormat, or GetOffsetsArray/GetConnectivityArray instead.) -- Deprecated since version 9.6.0.
  if dataset.GetVerts() and dataset.GetVerts().GetData().GetNumberOfTuples() > 0:
C:\Users\lukem\Python\mf-env\.venv\Lib\site-packages\trame_vtk\modules\vtk\serializers\data.py:49: DeprecationWarning: Call to deprecated method GetData. (Use ExportLegacyFormat, or GetOffsetsArray/GetConnectivityArray instead.) -- Deprecated since version 9.6.0.
  if dataset.GetLines() and dataset.GetLines().GetData().GetNumberOfTuples() > 0:
C:\Users\lukem\Python\mf-env\.venv\Lib\site-packages\trame_vtk\modules\vtk\serializers\data.py:55: DeprecationWarning: Call to deprecated method GetData. (Use ExportLegacyFormat, or GetOffsetsArray/GetConnectivityArray instead.) -- Deprecated since version 9.6.0.
  if dataset.GetPolys() and dataset.GetPolys

WindowsPath('C:/Users/lukem/Python/Projects/simple_modflow/examples/mf6/artifacts/canonical_prt_parallel/prt/prt.html')

## Lake-Safe Partitions

Partition cuts route around each physical lake. This avoids creating zero-area local LAK fragments while retaining contiguous model partitions.


In [6]:
partition_rows = []
for nparts in range(2, 9):
    candidate_mask = mf.canonical_partition_mask(model, nparts)
    prepared = model.parallel.split_model(
        workspace=root / f'split_{nparts}',
        mask=candidate_mask,
        write=False,
    )
    validation = prepared.validate()
    partition_rows.append({
        'partitions': nparts,
        'all_contiguous': bool(validation['contiguous'].all()),
        'minimum_columns': int(validation['columns'].min()),
        'maximum_columns': int(validation['columns'].max()),
    })
    print(f'Prepared and validated {nparts} contiguous partitions.')

partition_summary = pd.DataFrame(partition_rows).set_index('partitions')
assert partition_summary['all_contiguous'].all()
partition_summary


Prepared and validated 2 contiguous partitions.


Prepared and validated 3 contiguous partitions.


Prepared and validated 4 contiguous partitions.


Prepared and validated 5 contiguous partitions.


Prepared and validated 6 contiguous partitions.


Prepared and validated 7 contiguous partitions.


Prepared and validated 8 contiguous partitions.


,all_contiguous,minimum_columns,maximum_columns
partitions,,,
2,True,1250,1250
3,True,752,914
4,True,302,948
5,True,236,784
6,True,202,712
7,True,86,806
8,True,74,732


## Eight-Process MPI Run

The final partition set is written and run with eight MPI workers. The current Python environment's matching mf6 and mpiexec executables are discovered automatically.


In [7]:
mask = mf.canonical_partition_mask(model, 8)
split = model.parallel.split_model(workspace=root / 'split_8', mask=mask)
split.summary()


writing simulation...
  writing simulation name file...
  writing simulation tdis package...
  writing solution package ims_-1...
  writing package viz_prt_master_0_viz_prt_master_1...
  writing package viz_prt_master_0_viz_prt_master_1.mvr...
  writing package viz_prt_master_1_viz_prt_master_2...
  writing package viz_prt_master_1_viz_prt_master_2.mvr...
  writing package viz_prt_master_2_viz_prt_master_3...
  writing package viz_prt_master_2_viz_prt_master_3.mvr...
  writing package viz_prt_master_3_viz_prt_master_4...


  writing package viz_prt_master_3_viz_prt_master_4.mvr...
  writing package viz_prt_master_4_viz_prt_master_5...
  writing package viz_prt_master_4_viz_prt_master_5.mvr...
  writing package viz_prt_master_5_viz_prt_master_6...
  writing package viz_prt_master_5_viz_prt_master_6.mvr...
  writing package viz_prt_master_5_viz_prt_master_7...
  writing package viz_prt_master_5_viz_prt_master_7.mvr...
  writing package viz_prt_master_6_viz_prt_master_7...
  writing package viz_prt_master_6_viz_prt_master_7.mvr...
  writing model viz_prt_master_0...
    writing model name file...
    writing package disv...
    writing package ic...
    writing package npf...
    writing package sto...
    writing package chd...
    writing package rch...


    writing package sfr...
    writing package uzf...


    writing package oc...
  writing model viz_prt_master_1...
    writing model name file...
    writing package disv...
    writing package ic...
    writing package npf...
    writing package sto...
    writing package sfr...
    writing package uzf...


    writing package oc...
  writing model viz_prt_master_2...
    writing model name file...
    writing package disv...
    writing package ic...
    writing package npf...
    writing package sto...


...

    writing package lak...


    writing package sfr...
    writing package uzf...


    writing package oc...
    writing package mvr...
  writing model viz_prt_master_6...
    writing model name file...
    writing package disv...
    writing package ic...
    writing package npf...
    writing package sto...
    writing package sfr...
    writing package uzf...
    writing package oc...
  writing model viz_prt_master_7...
    writing model name file...
    writing package disv...
    writing package ic...
    writing package npf...
    writing package sto...
    writing package ghb...


    writing package sfr...
    writing package uzf...
    writing package oc...


,partition,columns,active_cells,active_fraction
0,0,313,1252,0.1252
1,1,313,1252,0.1252
2,2,313,1252,0.1252
3,3,313,1252,0.1252
4,4,273,1092,0.1092
5,5,732,2928,0.2928
6,6,74,296,0.0296
7,7,169,676,0.0676


In [8]:
parallel_success, parallel_report = split.run(processors=split.nparts, write=False, silent=True)
assert parallel_success, '\n'.join(parallel_report[-30:])
assert any('PARALLEL mode' in line for line in parallel_report)

pd.Series({
    'processors': split.nparts,
    'parallel_mode_confirmed': True,
    'termination': parallel_report[-1],
}, name='MPI run')


processors                                                  8
parallel_mode_confirmed                                  True
termination                 Normal termination of simulation.
Name: MPI run, dtype: object

## Reconstructed Results

Split outputs are reconstructed onto the original DISV grid and compared with the source-model heads. Small numerical differences are expected; large differences indicate a split or exchange problem.


In [9]:
head_comparison = split.compare_heads()
assert head_comparison.loc[0, 'max_absolute_error'] < 0.1, head_comparison
head_comparison


,count,mean_error,mean_absolute_error,max_absolute_error
0,10000,-0.000003,0.000004,0.000011


## Result

The same 50 x 50 canonical model now completes as a source GWF simulation, a finite MF6 PRT simulation, and an eight-process MPI split simulation. Every code-cell output above is part of the validation record.
